- Regimes were defined by KMeans, other notebook does this job "https://github.com/waseemshammout/trading_model/blob/main/Clustering/Draft_pipeline_05-08%20KMeans.ipynb"
- Focusing on Regime 0 which is 18% of the population
- Threshold optimization by profit
- hyperparameter tuning applied
- Feature selection: Shap based (non-zero important features are selected)
- Multiplicative interactions
- Training Period: 2022-07-01 - 2023-06-30 | Validation Period: 2023-07-01 - 2023-12-31 

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression 
from sklearn.preprocessing import StandardScaler
import catboost as cat 
import xgboost as xgb
import lightgbm as lgb
import shap
from datetime import timedelta, datetime
import warnings
warnings.filterwarnings("ignore")

## Input the 3 subsets of the regime (trainval, test24, test25) 
train_val_df = pd.read_csv('subset_r0.csv')           # here
test_df = pd.read_csv('test_r0.csv')                  # here
forward_test_df = pd.read_csv('forward_test_r0.csv')  # here
test_df['Time'] = pd.to_datetime(test_df['Time'])
forward_test_df['Time'] = pd.to_datetime(forward_test_df['Time'])
test_df = test_df[test_df['Time']>='2024-01-01']
train_val_df['Time'] = pd.to_datetime(train_val_df['Time'])
test_df = test_df.sort_values(by='Time', ascending=True)
forward_test_df = forward_test_df.sort_values(by='Time', ascending=True)


## By commenting the above block and un-commenting the below, we assign the full data of all regimes
# train_val_df = pd.read_csv('SE5.csv')
# train_val_df['Time'] = pd.to_datetime(train_val_df['Time'])
# train_val_df = train_val_df.sort_values(by='Time', ascending=True)
# test_df = train_val_df[train_val_df['Time']>='2024-01-01']
# train_val_df = train_val_df[train_val_df['Time']<'2024-01-01']
# forward_test_df = pd.read_csv('SE5-2.csv')
# forward_test_df['Time'] = pd.to_datetime(forward_test_df['Time'])
# forward_test_df = forward_test_df.sort_values('Time', ascending=True)

In [2]:
print(f"Train/Validation data shape: {train_val_df.shape}")
print(f"Test data shape: {test_df.shape}")
print(f"Forward Test data shape: {forward_test_df.shape}")

Train/Validation data shape: (7315, 718)
Test data shape: (421, 718)
Forward Test data shape: (214, 718)


In [3]:
# Helper Functions
def calculate_stability_score(y_true, pred, class1_proportion):
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    if fp == 0:
        return 1e9 if tp > 0 else 0.0
    if class1_proportion == 0:
        return 0.0
    stability = (tp / fp) / class1_proportion
    return stability

In [4]:
def evaluate_model(y_true, y_prob, model_name, class1_proportion_val, thresholds=None):
    """
    Simplified evaluation function with built-in optimal threshold finding,
    with a filter to avoid selecting unrealistic 100% precision/recall based on very few true positives.
    Returns metrics dictionary.
    """
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.99, 100)
        
    thresholds = [t for t in thresholds if 0.001 < t < 0.999]
    results = []
    
    # Find optimal threshold
    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(int)
        
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        stability = calculate_stability_score(y_true, y_pred, class1_proportion_val)
        
        results.append({
            'threshold': threshold,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'stability': stability,
            'tn': tn, # Store TP count for later filtering
            'fp': fp,
            'fn': fn,
            'tp': tp,
            'score': (tp * 800) - (fp * 200) - (fn * 200),
            'score_1': (tp * 800) - (fp * 200),
        })

    if not results:  # Fallback if no thresholds
        # This part remains as a safety fallback
        threshold = 0.5
        y_pred = (y_prob >= threshold).astype(int)
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        stability = calculate_stability_score(y_true, y_pred, class1_proportion_val)
        cm = confusion_matrix(y_true, y_pred)
    else:
        results_df = pd.DataFrame(results)

        misleading_options = (results_df['fp'] < 10) | (results_df['fn'] < 10) | (results_df['tn'] < 10)
        
        filtered_df = results_df[~(misleading_options)].copy()

        # Fallback if filtering removes all candidates (e.g., all predictions are extreme)
        if filtered_df.empty:
            print(f"Warning: All candidates filtered out for {model_name}. Reverting to original selection without extreme filtering.")
            filtered_df = results_df.copy() # Use original results if all are deemed extreme
        else:
            print(f"Extreme filtering is done, original results {results_df.shape[0]} row, filtered {filtered_df.shape[0]}")

        # Find best threshold from the filtered (or original) candidates
        # Prioritize max score_1, then max f1
        max_score_1 = filtered_df['score_1'].max()
        best_candidates = filtered_df[filtered_df['score_1'] == max_score_1]
        best_row = best_candidates.loc[best_candidates['f1'].idxmax()]
        
        threshold = best_row['threshold']
        precision = best_row['precision']
        recall = best_row['recall']
        f1 = best_row['f1']
        stability = best_row['stability']
        score = best_row['score']
        score_1 = best_row['score_1']
        
        # Re-calculate confusion matrix for the final selected optimal threshold
        y_pred = (y_prob >= threshold).astype(int)
        cm = confusion_matrix(y_true, y_pred)
        
    return {
        'model': model_name,
        'threshold': threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'stability': stability,
        'confusion_matrix': cm,
        'score': score,
        'score_1': score_1,
    }

In [5]:
n_features = 715

def preprocess_data(train, valid=None):

    train, valid = train.copy(), valid.copy()
    X_train, y_train = train.drop(columns=['Time','class']), train['class']
    X_valid, y_valid = valid.drop(columns=['Time','class']), valid['class']

    class_weight = y_train.value_counts()[0] / y_train.value_counts()[1]

    # Feature Selection (SHAP) on original + Regime feature, this SHAP calculation now includes the 'Regime' column
    tmp_model_final_shap = cat.CatBoostClassifier(random_state=1, scale_pos_weight=class_weight, loss_function='Logloss', eval_metric='F1', early_stopping_rounds=10, verbose=0)
    tmp_model_final_shap.fit(X_train, y_train, cat_features=[len(X_train.columns)-1], eval_set=[(X_valid, y_valid)])
    explainer_final = shap.TreeExplainer(tmp_model_final_shap)
    shap_values_final = explainer_final.shap_values(X_train)
    features_shap_final = pd.DataFrame({'Feature':X_train.columns,'Importance':np.abs(shap_values_final).mean(axis=0)}).sort_values(by='Importance', ascending=False)
    top_features = list(features_shap_final.loc[features_shap_final['Importance']>0,'Feature'].values)
    # top_features = list(features_shap_final.head(int(n_features * 0.02))['Feature'].values)

    X_train_selected = X_train[top_features]
    X_valid_selected = X_valid[top_features]

    # Feature Interactions (Multiplication on selected features)
    X_train_interacted = X_train_selected.copy()
    X_valid_interacted = X_valid_selected.copy()
    features_for_interaction = [f for f in top_features if f != 'Regime']

    for i in range(len(features_for_interaction)):
        for j in range(i + 1, len(features_for_interaction)):
            feature1 = features_for_interaction[i]
            feature2 = features_for_interaction[j]
            X_train_interacted[f'{feature1}_x_{feature2}'] = X_train_interacted[feature1] * X_train_interacted[feature2]
            X_valid_interacted[f'{feature1}_x_{feature2}'] = X_valid_interacted[feature1] * X_valid_interacted[feature2]

    return X_train_interacted, y_train, X_valid_interacted, y_valid, top_features

In [6]:
params_search_cat = {'depth': [4, 6, 8, 10],'learning_rate': [0.1, 0.05, 0.01, 0.005],'l2_leaf_reg': [1, 3, 5, 7, 9, 11],'bagging_temperature': [0, 0.5, 1, 3, 5],'subsample': [0.5, 0.7, 0.9, 1],'random_strength': [1e-9, 1e-6, 1e-3, 1e-1],'border_count': [32, 64, 128, 254]}
params_search_xgb = {'learning_rate': [0.1, 0.05, 0.01, 0.005],'max_depth': [3, 6, 10],'min_child_weight': [1, 3, 5],'subsample': [0.6, 0.8, 1.0],'colsample_bytree': [0.6, 0.8, 1.0],'gamma': [0, 0.1, 0.5],'reg_alpha': [0, 0.1, 1],'reg_lambda': [1, 5, 10]}
params_search_lgb = {'learning_rate': [0.1, 0.05, 0.01, 0.005],'max_depth': [3, 5, 7, 10, -1],'num_leaves': [15, 31, 63, 127],'min_child_samples': [10, 20, 30, 50],'subsample': [0.6, 0.8, 1.0],'colsample_bytree': [0.6, 0.8, 1.0],'reg_alpha': [0, 0.01, 0.1, 1],'reg_lambda': [0, 0.01, 0.1, 1],}

In [7]:
def train_base_model(X_train, y_train, X_valid=None, y_valid=None, model_name='CatBoost'):
    
    class_weight = y_train.value_counts()[0] / y_train.value_counts()[1]
    feature_scaler = None
    print(f'Hyperparameter tuning, base model: {model_name}, {datetime.now()}')
    if model_name == 'CatBoost':
        model = cat.CatBoostClassifier(random_state=1, scale_pos_weight=class_weight, loss_function='Logloss',  
                                       eval_metric='F1', early_stopping_rounds=50, iterations=1000, verbose=0)
        def cat_features_regime(cols):
            # This function is dedicated for catboost model.fit() to accomodate for Regime column as categorical feature
            if 'Regime' in cols:
                return ['Regime']
            return None
        search = RandomizedSearchCV(estimator=model, param_distributions=params_search_cat, n_iter=10, cv=TimeSeriesSplit(n_splits=3), random_state=1, scoring='f1', verbose=0, n_jobs=-1)
        search.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], cat_features=cat_features_regime(list(X_train.columns)))
    elif model_name == 'XGBoost':
        model = xgb.XGBClassifier(objective='binary:logistic', scale_pos_weight=class_weight, random_state=1, n_estimators=1000, early_stopping_rounds=10, enable_categorical=True, tree_method='hist', verbosity=0)
        search = RandomizedSearchCV(estimator=model, param_distributions=params_search_xgb, n_iter=10, cv=TimeSeriesSplit(n_splits=3), random_state=1, scoring='f1', verbose=0, n_jobs=-1)
        search.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
    elif model_name == 'LogReg':
        feature_scaler = StandardScaler()
        x_tr_scaled = feature_scaler.fit_transform(X_train)
        model = LogisticRegression(max_iter=1000, random_state=1, solver='liblinear', class_weight={0: 1, 1: class_weight}, C=0.01, penalty='l1')
        model.fit(x_tr_scaled, y_train)
    elif model_name == 'LGBM':
        model = lgb.LGBMClassifier(class_weight='balanced', objective='binary', random_state=1, learning_rate= 0.01, verbosity=-1, n_estimators=1000)
        search = RandomizedSearchCV(estimator=model, param_distributions=params_search_lgb, n_iter=10, cv=TimeSeriesSplit(n_splits=3), random_state=1, scoring='f1', verbose=0, n_jobs=-1)
        search.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], eval_metric="binary_logloss", callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)])

    if model_name != 'LogReg':
        model = search.best_estimator_

    print(f'Base model {model_name} is completed, {datetime.now()}')
    try:
        prob = model.predict_proba(X_valid)[:, 1]
    except Exception as e:
        print(f"{model} failed: {e}. Filling with zeros.")
        prob = np.zeros(len(y_valid))

    return prob, model, feature_scaler

## Train base models on the full data (or partial)

In [8]:
tr_start_date = datetime.strptime("2022-07-01", "%Y-%m-%d")
tr_end_date = datetime.strptime("2023-06-30", "%Y-%m-%d")
va_start_date = tr_end_date + timedelta(days=1)
va_end_date = train_val_df["Time"].max()
training_data = train_val_df[(train_val_df["Time"] >= tr_start_date) & (train_val_df["Time"] <= tr_end_date)]
validation_data = train_val_df[(train_val_df["Time"] >= va_start_date) & (train_val_df["Time"] <= va_end_date)]

print(f"Training Period: {tr_start_date.strftime('%Y-%m-%d')} - {tr_end_date.strftime('%Y-%m-%d')} | Validation Period: {va_start_date.strftime('%Y-%m-%d')} - {va_end_date.strftime('%Y-%m-%d')} \n")

X_train_interacted, y_train, X_valid_interacted, y_valid, top_features = preprocess_data(training_data, validation_data)

cat_prob, cat_model, _ = train_base_model(X_train_interacted, y_train, X_valid_interacted, y_valid, model_name='CatBoost')
xgb_prob, xgb_model, _ = train_base_model(X_train_interacted, y_train, X_valid_interacted, y_valid, model_name='XGBoost')
lr_prob, lr_model, feature_scaler = train_base_model(X_train_interacted, y_train, X_valid_interacted, y_valid, model_name='LogReg')
lgb_prob, lgb_model, _ = train_base_model(X_train_interacted, y_train, X_valid_interacted, y_valid, model_name='LGBM')

###################################### Check valid metrics #########################################

class1_proportion = y_valid.value_counts(normalize=True)[1]

cat_metrics = evaluate_model(y_valid, cat_prob, "CatBoost", class1_proportion)
xgb_metrics = evaluate_model(y_valid, xgb_prob, "XGBoost", class1_proportion)
lr_metrics = evaluate_model(y_valid, lr_prob, "LogReg", class1_proportion)
lgb_metrics = evaluate_model(y_valid, lgb_prob, "LGBM", class1_proportion)

full_data_metrics = pd.DataFrame([cat_metrics, xgb_metrics, lr_metrics, lgb_metrics])

full_data_metrics.round(3)

Training Period: 2022-07-01 - 2023-06-30 | Validation Period: 2023-07-01 - 2023-12-04 

Hyperparameter tuning, base model: CatBoost, 2025-08-11 22:02:11.591823
Base model CatBoost is completed, 2025-08-11 22:02:23.956185
Hyperparameter tuning, base model: XGBoost, 2025-08-11 22:02:23.959198
Base model XGBoost is completed, 2025-08-11 22:02:24.368439
Hyperparameter tuning, base model: LogReg, 2025-08-11 22:02:24.372436
Base model LogReg is completed, 2025-08-11 22:02:24.381176
Hyperparameter tuning, base model: LGBM, 2025-08-11 22:02:24.384179
Base model LGBM is completed, 2025-08-11 22:02:32.156946


,model,threshold,precision,recall,f1,stability,confusion_matrix,score,score_1
0,CatBoost,0.535,0.0,0.0,0.0,0.0,"[[170, 0], [7, 0]]",-1400.0,0.0
1,XGBoost,0.782,0.0,0.0,0.0,0.0,"[[170, 0], [7, 0]]",-1400.0,0.0
2,LogReg,0.010,0.0,0.0,0.0,0.0,"[[118, 52], [7, 0]]",-11800.0,-10400.0
3,LGBM,0.901,0.0,0.0,0.0,0.0,"[[170, 0], [7, 0]]",-1400.0,0.0


In [9]:
####################################################################################################
# Training the meta learner
####################################################################################################
meta_X_train = pd.DataFrame(
    {
        "cat_prob": cat_prob,
        "xgb_prob": xgb_prob,
        "logreg_prob": lr_prob,
        "lgbm_prob": lgb_prob,
    }
)
meta_y_train = y_valid

final_meta_learner = LogisticRegression(max_iter=1000)
final_meta_learner.fit(meta_X_train, meta_y_train)

LogisticRegression(max_iter=1000)

## Testing on 2024 data

In [10]:
def preprocess_test_data(test_df, interval='3m'): 
    
    assert interval in ["3m", "6m", "1y"]
    start_date = "2024-01-01"

    if interval == "1y":
        end_date = "2024-12-31"
    elif interval == "6m":
        end_date = "2024-06-30"
    elif interval == "3m":
        end_date = "2024-03-31"

    test_df = test_df[(test_df["Time"] >= start_date) & (test_df["Time"] <= end_date)]

    x_test = test_df.drop(columns=["Time", "class"])
    y_test = test_df["class"]

    x_test_selected = x_test[top_features] # align top features with training set
    # applying features interaction on the selected ones
    x_test_interacted = x_test_selected.copy()
    features_for_interaction = [f for f in top_features if f != 'Regime'] # exclude Regime
    for i in range(len(features_for_interaction)):
            for j in range(i + 1, len(features_for_interaction)):
                feature1 = features_for_interaction[i]
                feature2 = features_for_interaction[j]
                x_test_interacted[f'{feature1}_x_{feature2}'] = x_test_interacted[feature1] * x_test_interacted[feature2]
    assert list(X_train_interacted.columns) == list(x_test_interacted.columns)

    return x_test_interacted, y_test

In [11]:
x_test_interacted, y_test = preprocess_test_data(test_df=test_df, interval='1y')
cat_pred_test = cat_model.predict_proba(x_test_interacted)[:, 1]
xgb_pred_test = xgb_model.predict_proba(x_test_interacted)[:, 1]
lr_pred_test = lr_model.predict_proba(feature_scaler.transform(x_test_interacted))[:, 1]
lgbm_pred_test = lgb_model.predict_proba(x_test_interacted)[:, 1]
meta_X_test = pd.DataFrame({"cat_prob": cat_pred_test,"xgb_prob": xgb_pred_test,"logreg_prob": lr_pred_test,"lgbm_prob": lgbm_pred_test})
meta_y_test = y_test
meta_pred_prob = final_meta_learner.predict_proba(meta_X_test)[:, 1]
class1_proportion_test = y_test.value_counts(normalize=True)[1]
test_metrics = evaluate_model(y_test, meta_pred_prob, 'Stack Ensemble', class1_proportion_test)
optimal_thresh, p, r, f, stability = test_metrics['threshold'], test_metrics['precision'], test_metrics['recall'], test_metrics['f1'], test_metrics['stability']
final_pred = (meta_pred_prob > optimal_thresh).astype(int)

# Metrics
pd.DataFrame([test_metrics])

Extreme filtering is done, original results 100 row, filtered 1


,model,threshold,precision,recall,f1,stability,confusion_matrix,score,score_1
0,Stack Ensemble,0.039697,0.27907,0.25,0.263736,3.395161,"[[342, 31], [36, 12]]",-3800.0,3400.0


In [12]:
cat_metrics_test = evaluate_model(y_test, cat_pred_test, 'CatBoost', class1_proportion_test)
xgb_metrics_test = evaluate_model(y_test, xgb_pred_test, 'XGBoost', class1_proportion_test)
lr_metrics_test = evaluate_model(y_test, lr_pred_test, 'LogReg', class1_proportion_test)
lgbm_metrics_test = evaluate_model(y_test, lgbm_pred_test, 'LGBM', class1_proportion_test)

pd.DataFrame([cat_metrics_test,xgb_metrics_test,lr_metrics_test,lgbm_metrics_test])

Extreme filtering is done, original results 100 row, filtered 9
Extreme filtering is done, original results 100 row, filtered 71
Extreme filtering is done, original results 100 row, filtered 14
Extreme filtering is done, original results 100 row, filtered 80


,model,threshold,precision,recall,f1,stability,confusion_matrix,score,score_1
0,CatBoost,0.445556,0.132075,0.145833,0.138614,1.334692,"[[327, 46], [41, 7]]",-11800.0,-3600.0
1,XGBoost,0.475253,0.000000,0.000000,0.000000,0.000000,"[[361, 12], [48, 0]]",-12000.0,-2400.0
2,LogReg,0.514848,0.000000,0.000000,0.000000,0.000000,"[[360, 13], [48, 0]]",-12200.0,-2600.0
3,LGBM,0.653434,0.000000,0.000000,0.000000,0.000000,"[[361, 12], [48, 0]]",-12000.0,-2400.0


## Forward Test Data 2025

In [13]:
def preprocess_forward_test_data(test_df, interval='3m'):  # ["3m", "6m", "1y"]
    
    assert interval in ["3m", "6m", "1y"]
    start_date = test_df['Time'].min()

    if interval == "1y":
        end_date = start_date + timedelta(days=365)
    elif interval == "6m":
        end_date = start_date + timedelta(days=180)
    elif interval == "3m":
        end_date = start_date + timedelta(days=90)

    test_df = test_df[(test_df["Time"] >= start_date) & (test_df["Time"] <= end_date)]

    x_test = test_df.drop(columns=["Time", "class"])
    y_test = test_df["class"]

    x_test_selected = x_test[top_features] # align top features with training set
    # applying features interaction on the selected ones
    x_test_interacted = x_test_selected.copy()
    features_for_interaction = [f for f in top_features if f != 'Regime'] # exclude Regime
    for i in range(len(features_for_interaction)):
            for j in range(i + 1, len(features_for_interaction)):
                feature1 = features_for_interaction[i]
                feature2 = features_for_interaction[j]
                x_test_interacted[f'{feature1}_x_{feature2}'] = x_test_interacted[feature1] * x_test_interacted[feature2]
    assert list(X_train_interacted.columns) == list(x_test_interacted.columns)

    return x_test_interacted, y_test

In [14]:
x_test_interacted, y_test = preprocess_forward_test_data(test_df=forward_test_df, interval='1y')
cat_pred_test = cat_model.predict_proba(x_test_interacted)[:, 1]
xgb_pred_test = xgb_model.predict_proba(x_test_interacted)[:, 1]
lr_pred_test = lr_model.predict_proba(feature_scaler.transform(x_test_interacted))[:, 1]
lgbm_pred_test = lgb_model.predict_proba(x_test_interacted)[:, 1]
meta_X_test = pd.DataFrame({"cat_prob": cat_pred_test,"xgb_prob": xgb_pred_test,"logreg_prob": lr_pred_test,"lgbm_prob": lgbm_pred_test})
meta_y_test = y_test
meta_pred_prob = final_meta_learner.predict_proba(meta_X_test)[:, 1]
class1_proportion_test = y_test.value_counts(normalize=True)[1]
test_metrics = evaluate_model(y_test, meta_pred_prob, 'Stack Ensemble', class1_proportion_test)
optimal_thresh, p, r, f, stability = test_metrics['threshold'], test_metrics['precision'], test_metrics['recall'], test_metrics['f1'], test_metrics['stability']
final_pred = (meta_pred_prob > optimal_thresh).astype(int)

# Metrics
pd.DataFrame([test_metrics])

Extreme filtering is done, original results 100 row, filtered 1


,model,threshold,precision,recall,f1,stability,confusion_matrix,score,score_1
0,Stack Ensemble,0.039697,0.052632,0.15,0.077922,0.594444,"[[140, 54], [17, 3]]",-11800.0,-8400.0


In [15]:
cat_metrics_test = evaluate_model(y_test, cat_pred_test, 'CatBoost', class1_proportion_test)
xgb_metrics_test = evaluate_model(y_test, xgb_pred_test, 'XGBoost', class1_proportion_test)
lr_metrics_test = evaluate_model(y_test, lr_pred_test, 'LogReg', class1_proportion_test)
lgbm_metrics_test = evaluate_model(y_test, lgbm_pred_test, 'LGBM', class1_proportion_test)

pd.DataFrame([cat_metrics_test,xgb_metrics_test,lr_metrics_test,lgbm_metrics_test])

Extreme filtering is done, original results 100 row, filtered 2
Extreme filtering is done, original results 100 row, filtered 7
Extreme filtering is done, original results 100 row, filtered 6


,model,threshold,precision,recall,f1,stability,confusion_matrix,score,score_1
0,CatBoost,0.445556,0.0,0.0,0.0,0.0,"[[145, 49], [20, 0]]",-13800.0,-9800.0
1,XGBoost,0.118889,0.0,0.0,0.0,0.0,"[[183, 11], [20, 0]]",-6200.0,-2200.0
2,LogReg,0.455455,0.0,0.0,0.0,0.0,"[[178, 16], [20, 0]]",-7200.0,-3200.0
3,LGBM,0.108990,0.0,0.0,0.0,0.0,"[[194, 0], [20, 0]]",-4000.0,0.0


## 2008-2025 Predictions

In [16]:
df_2008_2024 = pd.read_csv('SE5.csv')
df_2025 = pd.read_csv('SE5-2.csv')
extended_data = pd.concat([df_2008_2024, df_2025], ignore_index=True)
extended_data['Time'] = pd.to_datetime(extended_data['Time'])
extended_data = extended_data.sort_values(by='Time', ascending=True)

In [17]:
def preprocess_full_data(full_df, start_date, end_date):
    
    full_df = full_df[(full_df["Time"] >= start_date) & (full_df["Time"] <= end_date)]

    x_full = full_df.drop(columns=["Time", "class"])
    y_full = full_df["class"]

    x_full_selected = x_full[top_features] # align top features with training set
    # applying features interaction on the selected ones
    x_full_interacted = x_full_selected.copy()
    features_for_interaction = [f for f in top_features if f != 'Regime'] # exclude Regime
    for i in range(len(features_for_interaction)):
            for j in range(i + 1, len(features_for_interaction)):
                feature1 = features_for_interaction[i]
                feature2 = features_for_interaction[j]
                x_full_interacted[f'{feature1}_x_{feature2}'] = x_full_interacted[feature1] * x_full_interacted[feature2]
    assert list(X_train_interacted.columns) == list(x_full_interacted.columns)

    return x_full_interacted, y_full




x_full_interacted, y_full = preprocess_full_data(full_df=extended_data, start_date='2008-01-01', end_date='2025-12-31')
x_full_interacted = x_full_interacted.astype(float)
y_full = y_full.astype(float)
cat_pred_full = cat_model.predict_proba(x_full_interacted)[:, 1]
xgb_pred_full = xgb_model.predict_proba(x_full_interacted)[:, 1]
lr_pred_full = lr_model.predict_proba(feature_scaler.transform(x_full_interacted))[:, 1]
lgbm_pred_full = lgb_model.predict_proba(x_full_interacted)[:, 1]
meta_X_full = pd.DataFrame({"cat_prob": cat_pred_full,"xgb_prob": xgb_pred_full,"logreg_prob": lr_pred_full,"lgbm_prob": lgbm_pred_full})
meta_y_full = y_full
meta_pred_prob = final_meta_learner.predict_proba(meta_X_full)[:, 1]
class1_proportion_full = y_full.value_counts(normalize=True)[1]
full_metrics = evaluate_model(y_full, meta_pred_prob, 'Stack Ensemble', class1_proportion_full)
optimal_thresh, p, r, f, stability = full_metrics['threshold'], full_metrics['precision'], full_metrics['recall'], full_metrics['f1'], full_metrics['stability']
final_pred = (meta_pred_prob > optimal_thresh).astype(int)

# Metrics
pd.DataFrame([full_metrics])

Extreme filtering is done, original results 100 row, filtered 4


,model,threshold,precision,recall,f1,stability,confusion_matrix,score,score_1
0,Stack Ensemble,0.019899,0.203886,0.817677,0.326388,1.238614,"[[5906, 29297], [1673, 7503]]",-191600.0,143000.0


In [18]:
cat_metrics_full = evaluate_model(y_full, cat_pred_full, 'CatBoost', class1_proportion_full)
xgb_metrics_full = evaluate_model(y_full, xgb_pred_full, 'XGBoost', class1_proportion_full)
lr_metrics_full = evaluate_model(y_full, lr_pred_full, 'LogReg', class1_proportion_full)
lgbm_metrics_full = evaluate_model(y_full, lgbm_pred_full, 'LGBM', class1_proportion_full)

pd.DataFrame([cat_metrics_full,xgb_metrics_full,lr_metrics_full,lgbm_metrics_full])

Extreme filtering is done, original results 100 row, filtered 10
Extreme filtering is done, original results 100 row, filtered 89
Extreme filtering is done, original results 100 row, filtered 65
Extreme filtering is done, original results 100 row, filtered 80


,model,threshold,precision,recall,f1,stability,confusion_matrix,score,score_1
0,CatBoost,0.445556,0.220189,0.510135,0.307606,1.365622,"[[18625, 16578], [4495, 4681]]",-469800.0,429200.0
1,XGBoost,0.069394,0.213093,0.669028,0.323233,1.309695,"[[12533, 22670], [3037, 6139]]",-230200.0,377200.0
2,LogReg,0.316869,0.207104,0.974063,0.341582,1.263273,"[[984, 34219], [238, 8938]]",259000.0,306600.0
3,LGBM,0.138687,0.221296,0.361486,0.274529,1.374435,"[[23531, 11672], [5859, 3317]]",-852600.0,319200.0


In [19]:
pd.DataFrame(
    {
        "CAT": (cat_pred_full > 0.445556).astype(int),
        "XGB": (xgb_pred_full > 0.069394).astype(int),
        "LR": (lr_pred_full > 0.316869).astype(int),
        "LGBM": (lgbm_pred_full > 0.138687).astype(int),
        "Stack Ensemble": final_pred,
        "Time": extended_data["Time"],
        'True class': extended_data['class']
    }
).to_csv("2008-2025_bin_pred_Regime0based.csv")